# M09b · Moindres carrés et régularisation — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m09b_moindres_carres/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m09b_moindres_carres/03_notebook.ipynb`, créée par `python tools/start_chapter.py M09b`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M09b.1 à M09b.12, M09b.20, M09b.21 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M09b.26 | Équations normales et résolution, avec ou sans ridge · `normal_equations, solve_least_squares` | 🔨 | ★ | 25 |
|  | M09b.27 | La matrice chapeau et les trois sommes de carrés · `hat_matrix, residual_decomposition` | 🔨 | ★ | 20 |
|  | M09b.28 | Compter les monômes, et la loss de Huber · `count_monomials, huber_loss` | 🔨 | ★ | 15 |
|  | M09b.29 | `np.linalg.lstsq` et scikit-learn contre tes équations normales | 📦 | ★ | 15 |
| B | M09b.30 | Prédire : le R² d'entraînement quand on ajoute une feature de pur bruit | 🔮 | ★ | 10 |
|  | M09b.31 | Simuler l'estimateur rétréci c·x̄ : le c qui minimise l'erreur quadratique | 🔬 | ★★ | 20 |
|  | M09b.32 | Le conditionnement de XᵀX avec des puissances non standardisées | 🔬 | ★★ | 20 |
|  | M09b.33 | Les lignes de niveau de la loss des moindres carrés en dimension 2, et son gradient | 🎨 | ★ | 20 |
| C | M09b.34 | Chasse à l'erreur : XXᵀ au lieu de XᵀX, un gradient de mauvais signe, un R² sans centrer | 🐛 | ★ | 15 |
|  | M09b.35 | Vérifier un gradient matriciel avec SymPy et par différences finies | 🛠️ | ★ | 15 |
|  | M09b.36 | Défi : approcher une fonction par un polynôme de degré au plus 5, avec une erreur maximale sous 1e−3 | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09b")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : losses et $R^2$, sommes de carrés, gradients, équations normales, colonne de 1, matrice chapeau, monômes, ridge et MAP, pénalités L1 et L2, log-vraisemblance, estimateur rétréci, un pas de descente de gradient, la standardisation avant ridge.
- Écrire les six fonctions de `mymath/linreg.py` et les tester, puis comparer avec `lstsq` et scikit-learn.
- Prédire l'effet de features de bruit, simuler un estimateur rétréci, mesurer un conditionnement, tracer une loss, chasser des erreurs, vérifier un gradient, approcher une fonction.

**Rappel express.** $\mathrm{MSE} = \frac{1}{n}\lVert \mathbf{r} \rVert^2$, $R^2 = 1 - \frac{\mathrm{SCR}}{\mathrm{SCT}}$ ; $\nabla(\mathbf{a}^\top\mathbf{w}) = \mathbf{a}$, $\nabla(\mathbf{w}^\top\mathbf{A}\mathbf{w}) = (\mathbf{A} + \mathbf{A}^\top)\mathbf{w}$, $\nabla\lVert \mathbf{y} - \mathbf{X}\mathbf{w} \rVert^2 = -2\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\mathbf{w})$ ; $\mathbf{X}^\top\mathbf{X}\,\mathbf{w} = \mathbf{X}^\top\mathbf{y}$ ; $\mathbf{H} = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$ ; $\binom{p + d}{d}$ monômes ; ridge : $(\mathbf{X}^\top\mathbf{X} + \lambda\mathbf{I})\mathbf{w} = \mathbf{X}^\top\mathbf{y}$, $\lambda = \sigma^2/\tau^2$ ; erreur = biais² + variance + bruit.

## Partie 0 · Vérifier tes exercices papier (✏️ M09b.1 à M09b.12, 🔗 M09b.20 et M09b.21)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"4/11"` ;
- une liste `[2, -1, 0]` pour un vecteur ou une forme, une liste de lignes `[[1, 0], [-2, 3]]` pour une matrice, `True` / `False`, ou des lettres entre guillemets, `"AC"`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M09b.1 — MSE, RMSE, MAE et loss de Huber de quatre résidus**

In [ ]:
# Ex M09b.1: the values you found on paper
answer_M09b_1a = ...  # a) a list of 4 numbers
answer_M09b_1b = ...  # b) a number (4 decimals)
answer_M09b_1c = ...  # c) a number (3 decimals)
answer_M09b_1d = ...  # d) a number (5 decimals)
answer_M09b_1e = ...  # e) a number (5 decimals)
answer_M09b_1f = ...  # f) a number (4 decimals)
answer_M09b_1g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M09b_1a, answer_M09b_1b, answer_M09b_1c, answer_M09b_1d, answer_M09b_1e, answer_M09b_1f, answer_M09b_1g]):
    wbm.check(f"M09b.1{letter}", answer)

**Ex M09b.2 — La décomposition SCT = SCE + SCR et le R² d'un ajustement donné**

In [ ]:
# Ex M09b.2: the values you found on paper
answer_M09b_2a = ...  # a) a number
answer_M09b_2b = ...  # b) a number
answer_M09b_2c = ...  # c) a number (2 decimals)
answer_M09b_2d = ...  # d) a number (2 decimals)
answer_M09b_2e = ...  # e) a number (3 decimals)
answer_M09b_2f = ...  # f) a list of 2 numbers (1 decimal)
answer_M09b_2g = ...  # g) an irreducible fraction, as text

for letter, answer in zip("abcdefg", [answer_M09b_2a, answer_M09b_2b, answer_M09b_2c, answer_M09b_2d, answer_M09b_2e, answer_M09b_2f, answer_M09b_2g]):
    wbm.check(f"M09b.2{letter}", answer)

**Ex M09b.3 — Les gradients de aᵀw, wᵀAw et ‖y − Xw‖², composante par composante en dimension 2**

In [ ]:
# Ex M09b.3: the values you found on paper
answer_M09b_3a = ...  # a) a number
answer_M09b_3b = ...  # b) a list of 2 numbers
answer_M09b_3c = ...  # c) a number
answer_M09b_3d = ...  # d) a list of 2 numbers
answer_M09b_3e = ...  # e) a list of 3 numbers
answer_M09b_3f = ...  # f) a number
answer_M09b_3g = ...  # g) a list of 2 numbers

for letter, answer in zip("abcdefg", [answer_M09b_3a, answer_M09b_3b, answer_M09b_3c, answer_M09b_3d, answer_M09b_3e, answer_M09b_3f, answer_M09b_3g]):
    wbm.check(f"M09b.3{letter}", answer)

**Ex M09b.4 — Les équations normales d'un modèle à deux features, sans ordonnée à l'origine**

In [ ]:
# Ex M09b.4: the values you found on paper
answer_M09b_4a = ...  # a) a list of 2 integers
answer_M09b_4b = ...  # b) a matrix
answer_M09b_4c = ...  # c) a list of 2 numbers
answer_M09b_4d = ...  # d) a list of 2 numbers (2 decimals)
answer_M09b_4e = ...  # e) a list of 4 numbers (2 decimals)
answer_M09b_4f = ...  # f) a list of 2 numbers (2 decimals)
answer_M09b_4g = ...  # g) a number (2 decimals)

for letter, answer in zip("abcdefg", [answer_M09b_4a, answer_M09b_4b, answer_M09b_4c, answer_M09b_4d, answer_M09b_4e, answer_M09b_4f, answer_M09b_4g]):
    wbm.check(f"M09b.4{letter}", answer)

**Ex M09b.5 — La colonne de 1 : l'ordonnée à l'origine et des résidus de moyenne nulle**

In [ ]:
# Ex M09b.5: the values you found on paper
answer_M09b_5a = ...  # a) a matrix
answer_M09b_5b = ...  # b) a list of 2 numbers
answer_M09b_5c = ...  # c) a list of 2 numbers (2 decimals)
answer_M09b_5d = ...  # d) a list of 4 numbers (2 decimals)
answer_M09b_5e = ...  # e) a number (2 decimals)
answer_M09b_5f = ...  # f) a list of 2 numbers
answer_M09b_5g = ...  # g) a list of 2 numbers (2 decimals)

for letter, answer in zip("abcdefg", [answer_M09b_5a, answer_M09b_5b, answer_M09b_5c, answer_M09b_5d, answer_M09b_5e, answer_M09b_5f, answer_M09b_5g]):
    wbm.check(f"M09b.5{letter}", answer)

**Ex M09b.6 — La matrice chapeau d'un petit tableau : H² = H et sa trace**

In [ ]:
# Ex M09b.6: the values you found on paper
answer_M09b_6a = ...  # a) a matrix
answer_M09b_6b = ...  # b) a matrix of integers
answer_M09b_6c = ...  # c) a matrix of integers
answer_M09b_6d = ...  # d) a number
answer_M09b_6e = ...  # e) an integer from 1 to 3
answer_M09b_6f = ...  # f) a list of 3 integers
answer_M09b_6g = ...  # g) a list of 3 integers

for letter, answer in zip("abcdefg", [answer_M09b_6a, answer_M09b_6b, answer_M09b_6c, answer_M09b_6d, answer_M09b_6e, answer_M09b_6f, answer_M09b_6g]):
    wbm.check(f"M09b.6{letter}", answer)

**Ex M09b.7 — Compter les monômes des features polynomiales**

In [ ]:
# Ex M09b.7: the values you found on paper
answer_M09b_7a = ...  # a) an integer
answer_M09b_7b = ...  # b) an integer
answer_M09b_7c = ...  # c) an integer
answer_M09b_7d = ...  # d) an integer
answer_M09b_7e = ...  # e) an integer
answer_M09b_7f = ...  # f) an integer
answer_M09b_7g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M09b_7a, answer_M09b_7b, answer_M09b_7c, answer_M09b_7d, answer_M09b_7e, answer_M09b_7f, answer_M09b_7g]):
    wbm.check(f"M09b.7{letter}", answer)

**Ex M09b.8 — Ridge en dimension 2 : résoudre (XᵀX + λI)w = Xᵀy pour trois valeurs de λ**

In [ ]:
# Ex M09b.8: the values you found on paper
answer_M09b_8a = ...  # a) a matrix of 2 rows and 3 columns
answer_M09b_8b = ...  # b) a list of 2 numbers
answer_M09b_8c = ...  # c) a list of 2 numbers (3 decimals)
answer_M09b_8d = ...  # d) a list of 2 numbers (2 decimals)
answer_M09b_8e = ...  # e) a list of 3 numbers (4 decimals)
answer_M09b_8f = ...  # f) a list of 2 numbers (2 decimals)
answer_M09b_8g = ...  # g) a list of 2 numbers

for letter, answer in zip("abcdefg", [answer_M09b_8a, answer_M09b_8b, answer_M09b_8c, answer_M09b_8d, answer_M09b_8e, answer_M09b_8f, answer_M09b_8g]):
    wbm.check(f"M09b.8{letter}", answer)

**Ex M09b.9 — Ridge comme MAP : un prior gaussien et λ = σ²/τ²**

In [ ]:
# Ex M09b.9: the values you found on paper
answer_M09b_9a = ...  # a) a number
answer_M09b_9b = ...  # b) a number
answer_M09b_9c = ...  # c) a list of 2 numbers (4 decimals)
answer_M09b_9d = ...  # d) a list of 2 numbers (4 decimals)
answer_M09b_9e = ...  # e) a number
answer_M09b_9f = ...  # f) a number (4 decimals)
answer_M09b_9g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M09b_9a, answer_M09b_9b, answer_M09b_9c, answer_M09b_9d, answer_M09b_9e, answer_M09b_9f, answer_M09b_9g]):
    wbm.check(f"M09b.9{letter}", answer)

**Ex M09b.10 — Les pénalités L1 et L2 de trois vecteurs de poids, et leurs boules**

In [ ]:
# Ex M09b.10: the values you found on paper
answer_M09b_10a = ...  # a) a list of 3 numbers
answer_M09b_10b = ...  # b) a list of 3 numbers
answer_M09b_10c = ...  # c) a letter: "u", "v" or "t"
answer_M09b_10d = ...  # d) letters, e.g. "AC"
answer_M09b_10e = ...  # e) a list of 2 numbers (4 decimals)
answer_M09b_10f = ...  # f) an integer
answer_M09b_10g = ...  # g) a list of 2 numbers (2 decimals)

for letter, answer in zip("abcdefg", [answer_M09b_10a, answer_M09b_10b, answer_M09b_10c, answer_M09b_10d, answer_M09b_10e, answer_M09b_10f, answer_M09b_10g]):
    wbm.check(f"M09b.10{letter}", answer)

**Ex M09b.11 — La log-vraisemblance gaussienne d'un modèle linéaire et l'estimation de σ²**

In [ ]:
# Ex M09b.11: the values you found on paper
answer_M09b_11a = ...  # a) a number (2 decimals)
answer_M09b_11b = ...  # b) a number (4 decimals)
answer_M09b_11c = ...  # c) a number (4 decimals)
answer_M09b_11d = ...  # d) a number (4 decimals)
answer_M09b_11e = ...  # e) a number (2 decimals)
answer_M09b_11f = ...  # f) a number (4 decimals)
answer_M09b_11g = ...  # g) a list of 2 numbers (2 decimals)

for letter, answer in zip("abcdefg", [answer_M09b_11a, answer_M09b_11b, answer_M09b_11c, answer_M09b_11d, answer_M09b_11e, answer_M09b_11f, answer_M09b_11g]):
    wbm.check(f"M09b.11{letter}", answer)

**Ex M09b.12 — L'estimateur rétréci c·x̄ : biais, variance et erreur quadratique**

In [ ]:
# Ex M09b.12: the values you found on paper
answer_M09b_12a = ...  # a) a number
answer_M09b_12b = ...  # b) a number (2 decimals)
answer_M09b_12c = ...  # c) a number (2 decimals)
answer_M09b_12d = ...  # d) a number
answer_M09b_12e = ...  # e) a number (2 decimals)
answer_M09b_12f = ...  # f) a number (2 decimals)
answer_M09b_12g = ...  # g) a number (2 decimals)

for letter, answer in zip("abcdefg", [answer_M09b_12a, answer_M09b_12b, answer_M09b_12c, answer_M09b_12d, answer_M09b_12e, answer_M09b_12f, answer_M09b_12g]):
    wbm.check(f"M09b.12{letter}", answer)

**Ex M09b.20 — Pont : la loss d'un modèle linéaire et son gradient pour un batch**

In [ ]:
# Ex M09b.20: the values you found on paper
answer_M09b_20a = ...  # a) a list of 4 numbers
answer_M09b_20b = ...  # b) a list of 4 numbers
answer_M09b_20c = ...  # c) a number (2 decimals)
answer_M09b_20d = ...  # d) a list of 3 numbers
answer_M09b_20e = ...  # e) a list of 3 numbers (2 decimals)
answer_M09b_20f = ...  # f) a list of 3 numbers (3 decimals)
answer_M09b_20g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M09b_20a, answer_M09b_20b, answer_M09b_20c, answer_M09b_20d, answer_M09b_20e, answer_M09b_20f, answer_M09b_20g]):
    wbm.check(f"M09b.20{letter}", answer)

**Ex M09b.21 — Pont : pourquoi standardiser les colonnes avant ridge**

In [ ]:
# Ex M09b.21: the values you found on paper
answer_M09b_21a = ...  # a) a list of 2 numbers (2 decimals)
answer_M09b_21b = ...  # b) a list of 2 numbers (2 decimals)
answer_M09b_21c = ...  # c) a list of 2 numbers (2 decimals)
answer_M09b_21d = ...  # d) a list of 2 numbers (4 decimals)
answer_M09b_21e = ...  # e) a list of 2 numbers (4 decimals)
answer_M09b_21f = ...  # f) a list of 2 numbers (2 decimals)
answer_M09b_21g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M09b_21a, answer_M09b_21b, answer_M09b_21c, answer_M09b_21d, answer_M09b_21e, answer_M09b_21f, answer_M09b_21g]):
    wbm.check(f"M09b.21{letter}", answer)

## Partie A · Ta librairie : équations normales, ridge, matrice chapeau, sommes de carrés, monômes, Huber ; NumPy et scikit-learn

Fiche §1, §3, §4 et §5. Les six fonctions de `mymath/linreg.py`, puis la même régression avec `np.linalg.lstsq` et scikit-learn.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def r2_score_np(y, y_hat):
    """The R² of the predictions y_hat, 1 - SSR / SST, with NumPy only (for the experiments)."""
    y, y_hat = np.asarray(y, dtype=float), np.asarray(y_hat, dtype=float)
    return float(1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2))

### Ex M09b.26 — Équations normales et résolution, avec ou sans ridge · `normal_equations, solve_least_squares` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer les équations normales et leur résolution, avec une pénalité ridge, et refuser un système singulier.  
**Prérequis :** fiche §3 et §5 · **mymath :** `linreg.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/linreg.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Ici, NumPy est permis en entier, `@`, `.T`, `np.eye` et `np.linalg.solve` compris : tu as écrit leurs équivalents en M09a. Sont interdits les solveurs qui font tout le travail : `np.linalg.lstsq`, `np.linalg.pinv`, `np.linalg.inv`, `scipy` et `sklearn` (les tests s'en servent comme oracles, et un test vérifie que tu ne les utilises pas). Ne modifie jamais les tableaux reçus.

Écris (fiche §3 et §5), en suivant les docstrings :
- `normal_equations(X, y)` : le couple $(\mathbf{X}^\top\mathbf{X}, \mathbf{X}^\top\mathbf{y})$, en tableaux NumPy de floats ; une `ValueError` si $\mathbf{X}$ n'est pas à deux dimensions, ou si $\mathbf{y}$ n'est pas un vecteur (à une dimension) d'une valeur par ligne de $\mathbf{X}$ : un $\mathbf{y}$ en colonne, de forme $(n, 1)$, est refusé ;
- `solve_least_squares(X, y, ridge=0.0)` : la solution de $(\mathbf{X}^\top\mathbf{X} + \lambda\mathbf{I})\mathbf{w} = \mathbf{X}^\top\mathbf{y}$, avec `np.linalg.solve` ; **tous** les poids sont pénalisés, celui d'une colonne de 1 compris. Une `ValueError` si les formes ne vont pas, si `ridge` est négatif, ou si la matrice est singulière (`np.linalg.solve` lève alors une `numpy.linalg.LinAlgError` : attrape-la).

Avec $\mathbf{X}$ = `[[1, 2], [1, -1], [1, 0], [1, 3]]` et $\mathbf{y}$ = `[4, 0, 2, 5]` (une colonne de 1, puis une feature), la vérification calcule :
- a) et b) les deux éléments du couple de `normal_equations(X, y)` (une matrice, puis une liste) ;
- c) `solve_least_squares(X, y)` (une liste, 6 décimales) ;
- d) `solve_least_squares(X, y, ridge=2)` (une liste, 6 décimales) ;
- e) le nom de l'erreur levée par `solve_least_squares([[1, 2], [2, 4], [3, 6]], [1, 2, 3])` ;
- f) `solve_least_squares([[1, 2], [2, 4], [3, 6]], [1, 2, 3], ridge=0.5)` (une liste, 4 décimales) ;
- g) le nom de l'erreur levée par `solve_least_squares([[1, 0], [0, 1]], [1, 2], ridge=-0.5)` ;
- h) le nom du type de la valeur rendue par `solve_least_squares`, en texte.

Dans ta copie : calcule a), b) et c) à la main. Pourquoi e) est-elle singulière, et pourquoi f) ne l'est-elle plus ? Vérifie que la solution de f) est proportionnelle à $(1, 2)$ : pourquoi ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09b")  # reload your saved file

In [ ]:
X_26, y_26 = [[1, 2], [1, -1], [1, 0], [1, 3]], [4, 0, 2, 5]
with checked("M09b.26a"):
    wbm.check("M09b.26a", np.asarray(mymath.linreg.normal_equations(X_26, y_26)[0]).tolist(), computed=True)
with checked("M09b.26b"):
    wbm.check("M09b.26b", np.asarray(mymath.linreg.normal_equations(X_26, y_26)[1]).tolist(), computed=True)
with checked("M09b.26c"):
    wbm.check("M09b.26c", np.asarray(mymath.linreg.solve_least_squares(X_26, y_26)).tolist(), computed=True)
with checked("M09b.26d"):
    wbm.check("M09b.26d", np.asarray(mymath.linreg.solve_least_squares(X_26, y_26, ridge=2)).tolist(), computed=True)
with checked("M09b.26e"):
    wbm.check("M09b.26e", raised(mymath.linreg.solve_least_squares, [[1, 2], [2, 4], [3, 6]], [1, 2, 3]), computed=True)
with checked("M09b.26f"):
    wbm.check("M09b.26f", np.asarray(mymath.linreg.solve_least_squares([[1, 2], [2, 4], [3, 6]], [1, 2, 3], ridge=0.5)).tolist(),
              computed=True)
with checked("M09b.26g"):
    wbm.check("M09b.26g", raised(mymath.linreg.solve_least_squares, [[1, 0], [0, 1]], [1, 2], ridge=-0.5),
              computed=True)
with checked("M09b.26h"):
    wbm.check("M09b.26h", type(mymath.linreg.solve_least_squares(X_26, y_26)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09b_linreg.py", "test_normal_equations_ or test_solve_least_squares_ or test_linreg_uses_")

### Ex M09b.27 — La matrice chapeau et les trois sommes de carrés · `hat_matrix, residual_decomposition` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la matrice chapeau sans inverse, et les trois sommes de carrés d'un ajustement.  
**Prérequis :** fiche §1 et §3 ; Ex M09b.26 · **mymath :** `linreg.py` · **Parcours :** C, E

Écris (fiche §1 et §3), en suivant les docstrings :
- `hat_matrix(X)` : $\mathbf{H} = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$, **sans** calculer d'inverse : résous $\mathbf{X}^\top\mathbf{X}\,\mathbf{M} = \mathbf{X}^\top$ (`np.linalg.solve` accepte une matrice comme second membre), puis $\mathbf{H} = \mathbf{X}\mathbf{M}$ ; une `ValueError` si $\mathbf{X}$ n'est pas à deux dimensions ou si $\mathbf{X}^\top\mathbf{X}$ est singulière ;
- `residual_decomposition(y, y_hat)` : le triplet $(\mathrm{SCT}, \mathrm{SCE}, \mathrm{SCR})$ (en anglais SST, SSE, SSR), autour de la moyenne de $\mathbf{y}$, en floats de Python ; une `ValueError` si les longueurs diffèrent ou sont nulles.

Avec $\mathbf{X}$ = `[[1, 0], [1, 1], [1, 2], [1, 3]]` (une droite sur les abscisses 0 à 3) et $\mathbf{y}$ = `[1, 0, 2, 5]`, la vérification calcule :
- a) la diagonale de `hat_matrix(X)`, les leviers (une liste, 6 décimales) ;
- b) la trace de `hat_matrix(X)` (6 décimales) ;
- c) $\hat{\mathbf{y}} = \mathbf{H}\mathbf{y}$ (une liste, 6 décimales) ;
- d) `residual_decomposition(y, H @ y)` (une liste de 3 nombres, 6 décimales) ;
- e) le $R^2$ qu'on en déduit (4 décimales) ;
- f) le nom de l'erreur levée par `residual_decomposition([1, 2], [1, 2, 3])` ;
- g) le nom du type du premier nombre rendu par `residual_decomposition`, en texte ;
- h) le nom de l'erreur levée par `hat_matrix([[1, 2], [2, 4], [3, 6]])`.

Dans ta copie : pourquoi les leviers des deux points extrêmes sont-ils les plus grands ? Vérifie que $\mathrm{SCT} = \mathrm{SCE} + \mathrm{SCR}$ en d) : pourquoi est-ce garanti ici ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09b")  # reload your saved file

In [ ]:
X_27, y_27 = np.array([[1, 0], [1, 1], [1, 2], [1, 3]], dtype=float), np.array([1, 0, 2, 5], dtype=float)
with checked("M09b.27a"):
    wbm.check("M09b.27a", np.diag(np.asarray(mymath.linreg.hat_matrix(X_27))).tolist(), computed=True)
with checked("M09b.27b"):
    wbm.check("M09b.27b", float(np.trace(np.asarray(mymath.linreg.hat_matrix(X_27)))), computed=True)
with checked("M09b.27c"):
    wbm.check("M09b.27c", (np.asarray(mymath.linreg.hat_matrix(X_27)) @ y_27).tolist(), computed=True)
with checked("M09b.27d"):
    sums_27 = mymath.linreg.residual_decomposition(y_27, np.asarray(mymath.linreg.hat_matrix(X_27)) @ y_27)
    wbm.check("M09b.27d", [float(s) for s in sums_27], computed=True)
with checked("M09b.27e"):
    sst_27, _, ssr_27 = mymath.linreg.residual_decomposition(y_27, np.asarray(mymath.linreg.hat_matrix(X_27)) @ y_27)
    wbm.check("M09b.27e", 1 - ssr_27 / sst_27, computed=True)
with checked("M09b.27f"):
    wbm.check("M09b.27f", raised(mymath.linreg.residual_decomposition, [1, 2], [1, 2, 3]), computed=True)
with checked("M09b.27g"):
    wbm.check("M09b.27g", type(mymath.linreg.residual_decomposition([1, 3], [2, 2])[0]).__name__, computed=True)
with checked("M09b.27h"):
    wbm.check("M09b.27h", raised(mymath.linreg.hat_matrix, [[1, 2], [2, 4], [3, 6]]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09b_linreg.py", "test_hat_matrix_ or test_residual_decomposition_ or test_linreg_uses_")

### Ex M09b.28 — Compter les monômes, et la loss de Huber · `count_monomials, huber_loss` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le nombre de colonnes des features polynomiales et la loss de Huber moyenne.  
**Prérequis :** fiche §1 et §4 · **mymath :** `linreg.py` · **Parcours :** C

Écris (fiche §1 et §4), en suivant les docstrings :
- `count_monomials(p, degree, include_constant=False)` : le nombre de monômes de degré au plus `degree` en `p` variables, $\binom{p + d}{d}$, moins 1 sans la constante, en `int` de Python (`math.comb` est permis) ; une `ValueError` si $p < 1$ ou si le degré est négatif ;
- `huber_loss(residuals, delta=1.0)` : la moyenne des $\ell_\delta(r_i)$ (fiche §1), en float de Python (`np.where` évite une boucle) ; une `ValueError` si la liste est vide ou si $\delta \le 0$.

La vérification calcule :
- a) `count_monomials(6, 3)` ;
- b) `count_monomials(6, 3, include_constant=True)` ;
- c) `count_monomials(2, 0)` ;
- d) `count_monomials(30, 4)` ;
- e) `huber_loss([1.5, -0.2, 3.0, -4.0], delta=2.0)` (6 décimales) ;
- f) `huber_loss([0.3, -0.1])` (6 décimales) ;
- g) le nom de l'erreur levée par `huber_loss([1.0], delta=0)` ;
- h) le nom du type de la valeur rendue par `count_monomials(3, 2)`, en texte.

Dans ta copie : calcule e) à la main, résidu par résidu. Que vaut c), et pourquoi ? Combien de colonnes ont les features polynomiales de degré 4 de 30 variables, et combien de nombres $\mathbf{X}^\top\mathbf{X}$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09b")  # reload your saved file

In [ ]:
with checked("M09b.28a"):
    wbm.check("M09b.28a", mymath.linreg.count_monomials(6, 3), computed=True)
with checked("M09b.28b"):
    wbm.check("M09b.28b", mymath.linreg.count_monomials(6, 3, include_constant=True), computed=True)
with checked("M09b.28c"):
    wbm.check("M09b.28c", mymath.linreg.count_monomials(2, 0), computed=True)
with checked("M09b.28d"):
    wbm.check("M09b.28d", mymath.linreg.count_monomials(30, 4), computed=True)
with checked("M09b.28e"):
    wbm.check("M09b.28e", mymath.linreg.huber_loss([1.5, -0.2, 3.0, -4.0], delta=2.0), computed=True)
with checked("M09b.28f"):
    wbm.check("M09b.28f", mymath.linreg.huber_loss([0.3, -0.1]), computed=True)
with checked("M09b.28g"):
    wbm.check("M09b.28g", raised(mymath.linreg.huber_loss, [1.0], delta=0), computed=True)
with checked("M09b.28h"):
    wbm.check("M09b.28h", type(mymath.linreg.count_monomials(3, 2)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09b_linreg.py", "test_count_monomials_ or test_huber_loss_ or test_linreg_uses_")

### Ex M09b.29 — `np.linalg.lstsq` et scikit-learn contre tes équations normales 📦 ★ ⏱️ 15 min
**Objectif :** faire la même régression avec `np.linalg.lstsq` et scikit-learn, lire ce que rend chaque fonction, et voir comment scikit-learn traite l'ordonnée à l'origine de ridge.  
**Prérequis :** fiche §3 et §5 ; Ex M09b.26 · **Parcours :** C, E

En pratique, on ne résout pas les équations normales à la main :
- `np.linalg.lstsq(X, y, rcond=None)` rend un quadruplet : la solution des moindres carrés, la somme des carrés des résidus, le **rang** de $\mathbf{X}$ et ses valeurs singulières (M12d) ; il ne forme jamais $\mathbf{X}^\top\mathbf{X}$ ;
- `sklearn.linear_model.LinearRegression()` ajoute lui-même l'ordonnée à l'origine (`fit_intercept=True` par défaut) : après `.fit(X, y)`, elle est dans `intercept_` et les autres poids dans `coef_` ;
- `sklearn.linear_model.Ridge(alpha=...)` minimise $\lVert \mathbf{y} - \mathbf{X}\mathbf{w} - b \rVert^2 + \alpha\lVert \mathbf{w} \rVert^2$ : l'ordonnée à l'origine $b$ n'est **pas** pénalisée.

La cellule fournie tire un dataset de 30 exemples à 2 features, `X_29` (sans colonne de 1) et `X1_29` (avec une colonne de 1 en premier), et les cibles `y_29`. Calcule :
- a) la solution de `np.linalg.lstsq(X1_29, y_29, rcond=None)` (une liste de 3 nombres, 4 décimales) ;
- b) `[intercept_, coef_[0], coef_[1]]` de `LinearRegression().fit(X_29, y_29)` (4 décimales) ;
- c) le rang que rend `np.linalg.lstsq` (un entier) ;
- d) `[intercept_, coef_[0], coef_[1]]` de `Ridge(alpha=5).fit(X_29, y_29)` (4 décimales) ;
- e) `mymath.linreg.solve_least_squares(X1_29, y_29, ridge=5)` (une liste, 4 décimales) ;
- f) vrai ou faux : l'ordonnée à l'origine de e) est plus proche de 0 que celle de d) (calcule-le à partir de tes réponses).

Dans ta copie : pourquoi a) et b) sont-ils égaux ? Pourquoi d) et e) diffèrent-ils, surtout sur l'ordonnée à l'origine, et pourquoi les deux autres poids changent-ils aussi ? Pourquoi `lstsq` est-il préférable aux équations normales quand des colonnes sont presque dépendantes ?

In [ ]:
rng_29 = np.random.default_rng(29)
x1_29, x2_29 = rng_29.uniform(0, 10, size=(2, 30))
y_29 = 2 + 0.5 * x1_29 - 1.2 * x2_29 + rng_29.normal(0, 1, 30)
X_29 = np.column_stack([x1_29, x2_29])                   # 30 examples, 2 features (no column of ones)
X1_29 = np.column_stack([np.ones(30), X_29])              # the same, with a column of ones first
print(X_29.shape, X1_29.shape, y_29[:3].round(3))

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

answer_M09b_29a = ...  # a) a list of 3 numbers (4 decimals)
answer_M09b_29b = ...  # b) a list of 3 numbers (4 decimals)
answer_M09b_29c = ...  # c) an integer
answer_M09b_29d = ...  # d) a list of 3 numbers (4 decimals)
answer_M09b_29e = ...  # e) a list of 3 numbers (4 decimals)
answer_M09b_29f = ...  # f) True or False, computed from d) and e)

In [ ]:
for letter, answer in zip("abcdef", [answer_M09b_29a, answer_M09b_29b, answer_M09b_29c, answer_M09b_29d,
                                     answer_M09b_29e, answer_M09b_29f]):
    wbm.check(f"M09b.29{letter}", answer)

## Partie B · Une prédiction, deux expériences et une figure

Fiche §1, §2, §4 et §8. Le $R^2$ et les features de bruit, l'estimateur rétréci, le conditionnement des puissances, les lignes de niveau de la loss.

### Ex M09b.30 — Prédire : le R² d'entraînement quand on ajoute une feature de pur bruit 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir l'effet de features sans aucun lien avec la cible sur le $R^2$ d'entraînement et sur celui de nouvelles données, puis le vérifier.  
**Prérequis :** fiche §1, §3 et §8 · **Parcours :** C, E

On ajuste par moindres carrés une droite ($\hat{y} = w_0 + w_1x$) à 20 exemples bruités, puis on ajoute à $\mathbf{X}$, une à une, des colonnes de **pur bruit** : des nombres tirés au hasard, sans aucun lien avec la cible. On mesure le $R^2$ sur les 20 exemples d'entraînement, et sur 1 000 nouveaux exemples tirés de la même façon. Choisis, pour chaque question, le numéro de la bonne réponse :
- a) chaque fois qu'on ajoute une colonne de bruit, le $R^2$ d'**entraînement** : (1) baisse ; (2) monte ou reste égal, jamais ne baisse ; (3) ne change pas ; (4) monte ou baisse selon le tirage ;
- b) combien de colonnes de bruit faut-il ajouter pour un $R^2$ d'entraînement exactement égal à 1 : (1) 2 ; (2) 10 ; (3) 18 ; (4) aucun nombre n'y suffit ;
- c) avec 18 colonnes de bruit, le $R^2$ sur les **nouvelles** données, comparé à celui de la droite seule : (1) plus grand ; (2) à peu près égal ; (3) nettement plus petit.

**Avant d'exécuter quoi que ce soit**, écris tes trois prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M09b_30a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M09b_30b = ...  # b) 1, 2, 3 or 4
prediction_M09b_30c = ...  # c) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abc", [prediction_M09b_30a, prediction_M09b_30b, prediction_M09b_30c]):
    wbm.check(f"M09b.30{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M09b_30a, prediction_M09b_30b, prediction_M09b_30c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    rng_30 = np.random.default_rng(30)
    x_train_30 = rng_30.uniform(0, 1, 20)
    y_train_30 = 1 + 2 * x_train_30 + rng_30.normal(0, 0.5, 20)
    x_test_30 = rng_30.uniform(0, 1, 1000)
    y_test_30 = 1 + 2 * x_test_30 + rng_30.normal(0, 0.5, 1000)
    noise_train_30, noise_test_30 = rng_30.normal(size=(20, 18)), rng_30.normal(size=(1000, 18))
    print(" k | R² train | R² test")
    for k in range(19):
        X_train = np.column_stack([np.ones(20), x_train_30, noise_train_30[:, :k]])
        X_test = np.column_stack([np.ones(1000), x_test_30, noise_test_30[:, :k]])
        w = np.linalg.lstsq(X_train, y_train_30, rcond=None)[0]
        print(f"{k:2d} | {r2_score_np(y_train_30, X_train @ w):8.4f} | {r2_score_np(y_test_30, X_test @ w):7.4f}")

**Après l'expérience**, dans ta copie : pourquoi le $R^2$ d'entraînement ne peut-il jamais baisser quand on ajoute une colonne (pense à ce que peut faire le poids de la nouvelle colonne) ? Que se passe-t-il pour $k = 18$ : combien de colonnes, combien d'exemples, combien de solutions a $\mathbf{X}\mathbf{w} = \mathbf{y}$ ? Pourquoi le $R^2$ de test, lui, baisse-t-il ? Quel terme de la fiche §8 grandit ?

### Ex M09b.31 — Simuler l'estimateur rétréci c·x̄ : le c qui minimise l'erreur quadratique 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer par simulation l'erreur quadratique d'un estimateur rétréci pour toute une grille de facteurs, et la comparer à la formule biais² + variance.  
**Prérequis :** fiche §8 ; Ex M09b.12 · **Parcours :** C

On observe $n = 5$ valeurs d'une loi normale de moyenne $\mu = 1$ et d'écart-type $\sigma = 3$, et l'on estime $\mu$ par $c\,\bar{x}$. La fiche (§8) et M09b.12 donnent l'erreur quadratique moyenne $E(c) = (c - 1)^2\mu^2 + c^2\frac{\sigma^2}{n}$. On la mesure ici par simulation. Écris :
- `simulated_mse(cs, mu, sigma, n, reps=200_000, seed=31)` : avec `rng = np.random.default_rng(seed)`, tire **une fois** un tableau d'échantillons `rng.normal(mu, sigma, size=(reps, n))`, calcule les `reps` moyennes $\bar{x}$ (une par ligne), puis rends, pour chaque $c$ de `cs`, la moyenne des $(c\,\bar{x} - \mu)^2$ (un tableau NumPy, un nombre par $c$) ;
- `formula_mse(c, mu, sigma, n)` : $E(c)$ par la formule.

La vérification calcule, avec la grille `cs = np.linspace(0, 1, 101)` :
- a) le $c$ de la grille où ta simulation est la plus petite, comparé au $c^*$ de la formule : vrai ou faux, ils diffèrent de moins de 0,02 ;
- b) vrai ou faux : sur toute la grille, ta simulation et ta formule diffèrent de moins de 0,02 (mais pas de 0 : une simulation n'est jamais exactement la formule) ;
- c) le $c^*$ exact de la formule, $\frac{\mu^2}{\mu^2 + \sigma^2/n}$ (4 décimales) ;
- d) le rapport $\frac{E(1)}{E(c^*)}$ (2 décimales).

Une cellule trace ensuite les deux courbes.

Dans ta copie : quel est le gain d'erreur quadratique de $c^*\bar{x}$ par rapport à $\bar{x}$ ? Pourquoi un $c$ plus petit que 1 aide-t-il surtout quand $\sigma^2/n$ est grand devant $\mu^2$ ? Quel est le lien avec ridge ?

In [ ]:
def simulated_mse(cs, mu, sigma, n, reps=200_000, seed=31):
    """The simulated mean squared error of c * x̄ as an estimator of mu, for each c of cs (one draw of samples)."""
    raise NotImplementedError("les échantillons une seule fois, les moyennes par ligne, puis une erreur par c")


def formula_mse(c, mu, sigma, n):
    """The mean squared error (c - 1)² mu² + c² sigma² / n of c * x̄."""
    raise NotImplementedError("biais² + variance")


answer_M09b_31c = ...  # c) the exact c* of the formula (4 decimals)
answer_M09b_31d = ...  # d) E(1) / E(c*) (2 decimals)

In [ ]:
CS_31 = np.linspace(0, 1, 101)
with checked("M09b.31a"):
    sim_31 = np.asarray(simulated_mse(CS_31, 1.0, 3.0, 5), dtype=float)
    c_star_31 = 1.0 / (1.0 + 9.0 / 5)
    wbm.check("M09b.31a", bool(sim_31.shape == CS_31.shape and abs(CS_31[np.argmin(sim_31)] - c_star_31) < 0.02),
              computed=True)
with checked("M09b.31b"):
    formula_31 = np.array([formula_mse(c, 1.0, 3.0, 5) for c in CS_31], dtype=float)
    sim_31 = np.asarray(simulated_mse(CS_31, 1.0, 3.0, 5), dtype=float)
    wbm.check("M09b.31b", bool(sim_31.shape == CS_31.shape and 0 < np.max(np.abs(sim_31 - formula_31)) < 0.02),
              computed=True)
wbm.check("M09b.31c", answer_M09b_31c)
wbm.check("M09b.31d", answer_M09b_31d)

**La figure** (elle utilise tes deux fonctions).

In [ ]:
with checked("M09b.31 figure"):
    sim_fig_31 = np.asarray(simulated_mse(CS_31, 1.0, 3.0, 5), dtype=float)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(CS_31, sim_fig_31, lw=3, alpha=0.6, label="simulation (200 000 échantillons)")
    ax.plot(CS_31, [formula_mse(c, 1.0, 3.0, 5) for c in CS_31], "k--", label="formule biais² + variance")
    ax.axvline(1.0 / (1.0 + 9.0 / 5), color="gray", lw=0.8)
    ax.set(xlabel="c", ylabel="erreur quadratique moyenne de c x̄", title="μ = 1, σ = 3, n = 5")
    ax.legend()
    plt.show()

### Ex M09b.32 — Le conditionnement de XᵀX avec des puissances non standardisées 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer le conditionnement de $\mathbf{X}^\top\mathbf{X}$ pour des features polynomiales, avec et sans mise à l'échelle de l'abscisse, et voir ce qu'il fait aux poids.  
**Prérequis :** fiche §4 ; Ex M09b.26 · **Parcours :** C

Le **conditionnement** d'une matrice carrée, `np.linalg.cond(A)`, dit à peu près par combien une petite erreur relative sur les données peut être multipliée dans la solution de $\mathbf{A}\mathbf{x} = \mathbf{b}$ (M12d en donnera la définition). Un `float64` a environ 16 chiffres significatifs : un conditionnement de $10^{16}$ peut tous les perdre. La cellule fournie définit 50 abscisses `x_32` entre 0 et 20, et les mêmes ramenées à $[-1, 1]$, `t_32`. Écris :
- `powers(x, degree)` : la matrice des colonnes $1, x, x^2, \ldots, x^{d}$ (forme $(n, d + 1)$) ;
- `cond_xtx(x, degree)` : le conditionnement de $\mathbf{X}^\top\mathbf{X}$ pour $\mathbf{X}$ = `powers(x, degree)`.

La vérification calcule :
- a) l'ordre de grandeur du conditionnement pour `x_32` et le degré 5 : la vérification compare l'exposant, `round(math.log10(...))` ;
- b) le même pour `t_32` et le degré 5 ;
- c) le plus petit degré de 1 à 8 pour lequel le conditionnement dépasse $10^{15}$ avec `x_32` ;
- d) vrai ou faux : avec `t_32`, il reste sous $10^6$ pour tous les degrés de 1 à 8.

Une dernière cellule modifie les cibles de $10^{-8}$ et compare la variation des poids et celle des prédictions, pour le degré 6.

Dans ta copie : pourquoi les colonnes $x^4$ et $x^5$ sur $[0, 20]$ sont-elles presque proportionnelles (fiche §4) ? Pourquoi les poids bougent-ils beaucoup, et les prédictions presque pas ? Quand cela devient-il grave ?

In [ ]:
x_32 = np.linspace(0, 20, 50)                  # 50 abscissas between 0 and 20
t_32 = (x_32 - 10) / 10                           # the same, scaled to [-1, 1]

In [ ]:
def powers(x, degree):
    """The matrix whose columns are 1, x, x², ..., x**degree (shape (n, degree + 1))."""
    raise NotImplementedError("une colonne par puissance, de 0 à degree")


def cond_xtx(x, degree):
    """The condition number of XᵀX for X = powers(x, degree)."""
    raise NotImplementedError("le conditionnement de XᵀX, par une fonction de np.linalg")

In [ ]:
with checked("M09b.32a"):
    if np.shape(powers(x_32, 5)) != (50, 6):
        print(f"❌ Ex M09b.32 : powers(x_32, 5) a la forme {np.shape(powers(x_32, 5))}, pas (50, 6) : une colonne par "
              "puissance, de x⁰ = 1 à x⁵.")
    wbm.check("M09b.32a", round(math.log10(max(cond_xtx(x_32, 5), 1e-300))), computed=True)
with checked("M09b.32b"):
    wbm.check("M09b.32b", round(math.log10(max(cond_xtx(t_32, 5), 1e-300))), computed=True)
with checked("M09b.32c"):
    wbm.check("M09b.32c", next((d for d in range(1, 9) if cond_xtx(x_32, d) > 1e15), 0), computed=True)
with checked("M09b.32d"):
    wbm.check("M09b.32d", bool(all(cond_xtx(t_32, d) < 1e6 for d in range(1, 9))), computed=True)

**Ce que cela fait aux poids** (une expérience fournie, avec NumPy).

In [ ]:
rng_32 = np.random.default_rng(32)
V6 = np.column_stack([x_32 ** k for k in range(7)])            # degree 6, raw powers
T6 = np.column_stack([t_32 ** k for k in range(7)])            # degree 6, scaled abscissas
target = np.sin(x_32 / 4)
nudged = target + 1e-8 * rng_32.normal(size=x_32.size)         # a change of 1e-8 in the data
for name, D in (("raw", V6), ("scaled", T6)):
    w0 = np.linalg.solve(D.T @ D, D.T @ target)
    w1 = np.linalg.solve(D.T @ D, D.T @ nudged)
    print(f"{name:6s} · largest relative change of a weight: {np.max(np.abs(w1 - w0) / np.abs(w0)):.1e}"
          f" · largest change of a prediction: {np.max(np.abs(D @ (w1 - w0))):.1e}")

### Ex M09b.33 — Les lignes de niveau de la loss des moindres carrés en dimension 2, et son gradient 🎨 ★ ⏱️ 20 min
**Objectif :** tracer la loss des moindres carrés comme une fonction des deux poids, avec son minimum et les directions de descente, et voir l'effet du centrage sur sa forme.  
**Prérequis :** fiche §2 et §3 ; M05b (lignes de niveau) · **Parcours :** C

Pour une droite $\hat{y} = w_0 + w_1x$, la loss $L(w_0, w_1) = \lVert \mathbf{y} - \mathbf{X}\mathbf{w} \rVert^2$ est une fonction de **deux** variables : on peut tracer ses lignes de niveau dans le plan des poids. Écris `draw_loss(ax, X, y, w0_lim, w1_lim)`, qui dessine dans l'axe `ax`, sur le rectangle des poids $[w_0^{\min}, w_0^{\max}] \times [w_1^{\min}, w_1^{\max}]$ :
- au moins 8 lignes de niveau de $L$ (`ax.contour`, sur une grille fine faite par `np.meshgrid`) ; des niveaux en progression géométrique au-dessus du minimum, `L_min + np.geomspace(...)`, montrent mieux la forme ;
- le minimum, la solution des équations normales, par un seul point (`ax.plot(w0, w1, "*")`) ;
- les flèches de la **direction de descente** $-\nabla L(\mathbf{w}) = 2\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\mathbf{w})$ sur une grille grossière d'au moins 16 points (`ax.quiver`).

La cellule de la figure s'en sert deux fois sur les cinq points fournis : avec l'abscisse $x$ brute, puis centrée ($x - \bar{x}$). Une cellule de vérification essaie ta fonction sur un autre dataset.

Dans ta copie : pourquoi les lignes de niveau sont-elles des ellipses (pense au degré de $L$) ? Pourquoi sont-elles si allongées et penchées avec $x$ brut, et alignées sur les axes avec $x$ centré (calcule $\mathbf{X}^\top\mathbf{X}$ dans les deux cas) ? Les flèches pointent-elles vers le minimum ? Qu'en déduis-tu pour une descente de gradient sur des données non centrées ?

In [ ]:
x_33 = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_33 = np.array([1.5, 2.0, 3.5, 3.0, 5.0])
X_33 = np.column_stack([np.ones(5), x_33])               # w = (w0, w1): ordinate at the origin, slope
Xc_33 = np.column_stack([np.ones(5), x_33 - x_33.mean()])  # the same with a centered abscissa

In [ ]:
def draw_loss(ax, X, y, w0_lim, w1_lim):
    """In ax: level lines of L(w) = ‖y - Xw‖² over the rectangle w0_lim × w1_lim, the minimum (one point), and the
    descent directions -∇L(w) on a coarse grid (quiver)."""
    raise NotImplementedError("une grille de poids, L sur la grille, ax.contour ; le minimum ; ax.quiver")

In [ ]:
with checked("M09b.33 figure"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    try:
        created = drawn(draw_loss, axes[0], X_33, y_33, (-2.5, 3.5), (0.0, 1.6))
        drawn(draw_loss, axes[1], Xc_33, y_33, (1.0, 5.0), (0.0, 1.6))
    except BaseException:
        plt.close(fig)
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M09b.33 : draw_loss ouvre sa propre figure : dessine dans l'axe reçu.")
    else:
        axes[0].set(title="x brut", xlabel="w0", ylabel="w1")
        axes[1].set(title="x centré", xlabel="w0", ylabel="w1")
        plt.show()

**Vérifications automatiques** (sur un autre dataset, dans des axes neufs).

In [ ]:
with checked("M09b.33 vérifications"):
    import matplotlib.contour as mcontour
    import matplotlib.quiver as mquiver
    problems = []
    X_t = np.array([[1.0, -1.0], [1.0, 0.5], [1.0, 2.0], [1.0, 3.0]])    # another dataset
    y_t = np.array([0.0, 2.0, 1.0, 4.0])
    w_t = np.linalg.solve(X_t.T @ X_t, X_t.T @ y_t)
    fig_t, ax_t = plt.subplots()
    try:
        created = drawn(draw_loss, ax_t, X_t, y_t, (w_t[0] - 2, w_t[0] + 2), (w_t[1] - 1, w_t[1] + 1))
    finally:
        plt.close(fig_t)
    if created:
        problems.append("draw_loss ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        contours = [c for c in ax_t.collections if isinstance(c, mcontour.ContourSet)]
        quivers = [c for c in ax_t.collections if isinstance(c, mquiver.Quiver)]
        points = [line for line in ax_t.get_lines() if len(line.get_xdata()) == 1]
        if not contours or len(contours[0].levels) < 8:
            problems.append("trace au moins 8 lignes de niveau de la loss (ax.contour)")
        if not any(np.allclose([line.get_xdata()[0], line.get_ydata()[0]], w_t, atol=1e-6) for line in points):
            problems.append("marque le minimum par un seul point (ax.plot(w0, w1, marker)), placé à la solution des "
                            "équations normales")
        if not quivers:
            problems.append("il manque les flèches de la descente (ax.quiver)")
        else:
            q = quivers[0]
            xy = np.asarray(q.get_offsets(), dtype=float)
            u, v = np.ravel(q.U), np.ravel(q.V)
            grads = np.array([-2 * X_t.T @ (y_t - X_t @ w) for w in xy])
            norms = np.hypot(grads[:, 0], grads[:, 1]) * np.hypot(u, v)
            keep = norms > 1e-12
            cosines = -(u * grads[:, 0] + v * grads[:, 1])[keep] / norms[keep]
            if xy.shape[0] < 16:
                problems.append("dessine au moins 16 flèches, sur une grille")
            elif cosines.size == 0 or np.min(cosines) < 0.999:
                problems.append("chaque flèche doit pointer dans la direction de descente, −∇L(w), en son point de "
                                "départ")
    if problems:
        print("❌ Ex M09b.33 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M09b.33 : lignes de niveau, minimum et flèches de descente justes.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1 à §4. Trois erreurs à trouver, la vérification d'un gradient, puis une approximation polynomiale sous une erreur maximale.

### Ex M09b.34 — Chasse à l'erreur : XXᵀ au lieu de XᵀX, un gradient de mauvais signe, un R² sans centrer 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques des moindres carrés, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1, §2 et §3 · **Parcours :** C

Un collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle doit calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
X = np.array([[1.0, 2.0], [1.0, -1.0], [1.0, 0.0]])    # 3 examples: a column of ones, then one feature
y = np.array([3.0, 0.0, 2.0])
w = np.array([1.0, 0.5])
G = X @ X.T                                   # (A) the 2 × 2 matrix XᵀX of the normal equations: G[0, 1]
r = y - X @ w                                 # (B) the residuals of the predictions Xw: r[0]
g = 2 * X.T @ (y - X @ w)                     # (C) the gradient of ‖y − Xw‖² at w: g[1]
w_new = w + 0.1 * 2 * X.T @ r                 # (D) one gradient descent step on ‖y − Xw‖², with η = 0.1: w_new[1]
r2 = 1 - np.sum(r ** 2) / np.sum(y ** 2)      # (E) the R² of the predictions Xw
mse = np.mean(r ** 2)                         # (F) the MSE of the predictions Xw
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 4 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? La ligne (A) s'exécute sans erreur : quelle est la forme de son résultat, et pourquoi Python ne signale-t-il rien ? La ligne (D) ajoute au lieu de retrancher : pourquoi est-elle juste ?

In [ ]:
answer_M09b_34a = ...  # a) letters, as text
answer_M09b_34b = ...  # b) a list of 6 values (4 decimals)

In [ ]:
wbm.check("M09b.34a", answer_M09b_34a)
wbm.check("M09b.34b", answer_M09b_34b)

### Ex M09b.35 — Vérifier un gradient matriciel avec SymPy et par différences finies 🛠️ ★ ⏱️ 15 min
**Objectif :** vérifier une formule de gradient de deux façons : numériquement, par des différences finies centrées, et exactement, avec SymPy.  
**Prérequis :** fiche §2 ; M05a (différences finies) ; Ex M09b.13 · **Parcours :** C

Une formule de gradient fausse ne fait pas planter le code : la descente converge mal, ou vers le mauvais point. Avant de s'en servir, on la vérifie : c'est la **vérification du gradient** (*gradient checking*). Écris :
- `numerical_gradient(f, w, h=1e-6)` : le gradient de `f` en `w` par différences finies **centrées**, $\frac{f(\mathbf{w} + h\mathbf{e}_k) - f(\mathbf{w} - h\mathbf{e}_k)}{2h}$ pour chaque coordonnée $k$ (un tableau NumPy) ;
- `gradient_error(f, grad, w)` : le plus grand écart, en valeur absolue, entre `numerical_gradient(f, w)` et `grad(w)`, divisé par $\max(1, \text{la plus grande valeur absolue du gradient numérique})$ ;
- `symbolic_check(p)` : avec SymPy, un vecteur $\mathbf{w}$ de $p$ symboles et une matrice $\mathbf{A}$ de $p^2$ symboles (`sympy.symbols`, `sympy.Matrix`), le gradient de $\mathbf{w}^\top\mathbf{A}\mathbf{w}$ (`sympy.diff` sur chaque $w_k$) ; rends `True` si sa différence avec $(\mathbf{A} + \mathbf{A}^\top)\mathbf{w}$, développée (`sympy.expand`), est nulle, en `bool` de Python.

La vérification essaie `numerical_gradient` sur une fonction de gradient connu, puis `gradient_error` sur $(\mathbf{A} + \mathbf{A}^\top)\mathbf{w}$ (juste), sur $2\mathbf{A}\mathbf{w}$ (faux, $\mathbf{A}$ n'étant pas symétrique) et sur le gradient des moindres carrés ; enfin `symbolic_check(3)`.

Dans ta copie : pourquoi des différences **centrées** plutôt que $\frac{f(\mathbf{w} + h\mathbf{e}_k) - f(\mathbf{w})}{h}$ (M05a) ? Pourquoi pas $h = 10^{-12}$ ? Pourquoi `gradient_error` divise-t-il par la taille du gradient ? Combien d'évaluations de `f` coûte un gradient numérique, et pourquoi ne s'en sert-on pas pour entraîner un réseau ?

In [ ]:
def numerical_gradient(f, w, h=1e-6):
    """The gradient of f at w by centered finite differences, one coordinate at a time."""
    raise NotImplementedError("pour chaque k : (f(w + h e_k) − f(w − h e_k)) / (2h)")


def gradient_error(f, grad, w):
    """max |numerical gradient − grad(w)|, divided by max(1, max |numerical gradient|)."""
    raise NotImplementedError("numerical_gradient, puis le plus grand écart relatif")


def symbolic_check(p):
    """True if the SymPy gradient of wᵀAw (w: p symbols, A: p² symbols) equals (A + Aᵀ)w."""
    raise NotImplementedError("sympy.symbols, sympy.Matrix, sympy.diff, puis sympy.expand de la différence")

In [ ]:
with checked("M09b.35 vérifications"):
    problems = []
    w_t = np.array([0.3, -1.2, 2.0])
    approx_t = np.asarray(numerical_gradient(lambda v: float(np.sum(np.sin(v))), w_t), dtype=float)
    if approx_t.shape != w_t.shape or not np.allclose(approx_t, np.cos(w_t), rtol=0, atol=1e-8):
        problems.append("numerical_gradient ne donne pas le gradient de sum(sin(w)), qui vaut cos(w) : différence "
                        "centrée (f(w + h e_k) − f(w − h e_k)) / (2h), une coordonnée à la fois")
    else:
        rng_t = np.random.default_rng(35)
        A_t, X_t, y_t = rng_t.normal(size=(3, 3)), rng_t.normal(size=(6, 3)), rng_t.normal(size=6)
        quad_t = lambda v: float(v @ A_t @ v)                                    # noqa: E731
        good_t = gradient_error(quad_t, lambda v: (A_t + A_t.T) @ v, w_t)
        bad_t = gradient_error(quad_t, lambda v: 2 * A_t @ v, w_t)
        ls_t = gradient_error(lambda v: float(np.sum((y_t - X_t @ v) ** 2)), lambda v: -2 * X_t.T @ (y_t - X_t @ v), w_t)
        print(f"(A + Aᵀ)w : {good_t:.1e} · 2Aw : {bad_t:.1e} · −2Xᵀ(y − Xw) : {ls_t:.1e}")
        if not (good_t < 1e-8 and ls_t < 1e-8):
            problems.append("gradient_error doit être minuscule pour un gradient juste : compare numerical_gradient(f, "
                            "w) et grad(w), puis divise le plus grand écart par max(1, la plus grande valeur absolue)")
        elif not bad_t > 1e-2:
            problems.append("gradient_error doit être grand pour 2Aw, faux quand A n'est pas symétrique")
    try:
        same = symbolic_check(3)
    except NotImplementedError:
        raise
    if same is not True:
        problems.append("symbolic_check(3) doit rendre True : le gradient de SymPy et (A + Aᵀ)w, simplifiés, doivent "
                        "coïncider (renvoie un bool de Python)")
    if problems:
        print("❌ Ex M09b.35 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M09b.35 : le gradient numérique, la mesure d'erreur et la vérification symbolique sont justes.")

### Ex M09b.36 — Défi : approcher une fonction par un polynôme de degré au plus 5, avec une erreur maximale sous 1e−3 🏆 ★★★ ⏱️ 25 min
**Objectif :** construire, avec tes moindres carrés, un polynôme de degré au plus 5 dont l'erreur maximale sur tout un intervalle passe sous $10^{-3}$.  
**Prérequis :** fiche §3 et §4 ; Ex M09b.26 et M09b.32 · **Parcours :** C

La fonction $f(x) = \frac{20}{20 + x}$ décroît de 1 à $\frac{1}{3}$ sur $[0, 40]$. **L'objectif** : un polynôme $p$ de degré au plus 5 tel que $\max_{x \in [0, 40]} |p(x) - f(x)| < 10^{-3}$, l'erreur étant mesurée sur la grille `GRID_36` de 4 001 points. Le polynôme s'écrit dans la variable $t = \frac{x - 20}{20}$, qui va de $-1$ à $1$ (M09b.32 dit pourquoi) : $p = c_0 + c_1t + \cdots + c_5t^5$.

Écris `fit_36()`, qui rend les coefficients $(c_0, \ldots, c_5)$ (au plus 6 nombres), calculés avec **ta** fonction `mymath.linreg.solve_least_squares` (sans `np.polyfit` ni `lstsq`) à partir de points $(x_i, f(x_i))$ que tu choisis. La cellule fournie définit `target_36`, `GRID_36` et `evaluate_36(coefficients, x)`, qui évalue ton polynôme.

Commence par le plus simple : les moindres carrés sur les 4 001 points de la grille. Mesure l'erreur maximale et regarde **où** elle est atteinte. La vérification affiche ton erreur maximale et l'endroit où elle est atteinte.

Dans ta copie : quelle erreur donnait ta première version, et où était-elle la plus grande ? Qu'as-tu changé ? Pourquoi les moindres carrés ne minimisent-ils pas l'erreur maximale ? Un degré 4 pourrait-il suffire (essaie) ?

In [ ]:
def target_36(x):
    """The function to approximate on [0, 40]."""
    return 20 / (20 + np.asarray(x, dtype=float))


GRID_36 = np.linspace(0, 40, 4001)                # where the maximal error is measured


def evaluate_36(coefficients, x):
    """The polynomial sum_k coefficients[k] * t**k, with t = (x - 20) / 20 (t goes from -1 to 1)."""
    t = (np.asarray(x, dtype=float) - 20) / 20
    return sum(c * t ** k for k, c in enumerate(coefficients))

In [ ]:
def fit_36():
    """At most 6 coefficients c_0, ..., c_5 of a polynomial in t = (x - 20) / 20 that approximates target_36 on
    [0, 40] with a maximal error below 1e-3, computed with mymath.linreg.solve_least_squares."""
    raise NotImplementedError("des points x_i, la matrice des puissances de t_i, puis solve_least_squares")

In [ ]:
with checked("M09b.36 défi"):
    ok_36 = False
    coefficients_36 = fit_36()
    coefficients_36 = np.asarray(coefficients_36, dtype=float).ravel()
    if coefficients_36.size == 0 or coefficients_36.size > 6:
        print(f"❌ Ex M09b.36 : {coefficients_36.size} coefficients : un polynôme de degré au plus 5 en a au plus 6.")
    elif not np.all(np.isfinite(coefficients_36)):
        print("❌ Ex M09b.36 : des coefficients non finis.")
    else:
        error_36 = float(np.max(np.abs(evaluate_36(coefficients_36, GRID_36) - target_36(GRID_36))))
        worst_36 = float(GRID_36[np.argmax(np.abs(evaluate_36(coefficients_36, GRID_36) - target_36(GRID_36)))])
        print(f"erreur maximale : {error_36:.2e}, atteinte en x = {worst_36:g}")
        if error_36 < 1e-3:
            ok_36 = True
            print("✅ Ex M09b.36 : défi réussi : l'erreur maximale est sous 1e-3.")
        else:
            print("❌ Ex M09b.36 : l'erreur maximale dépasse 1e-3. Regarde où elle est atteinte : les moindres carrés "
                  "minimisent une somme de carrés, pas l'erreur maximale.")
with checked("M09b.36"):
    fit_36()                                              # ⏳ while fit_36 is not written
    wbm.check("M09b.36", bool(ok_36), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une MSE, une MAE, une loss de Huber et un $R^2$, et dire ce que mesure chacun ?
2. Sais-tu calculer les gradients de $\mathbf{a}^\top\mathbf{w}$, de $\mathbf{w}^\top\mathbf{A}\mathbf{w}$ et de $\lVert \mathbf{y} - \mathbf{X}\mathbf{w} \rVert^2$, et en tirer les équations normales ?
3. Sais-tu expliquer les moindres carrés comme une projection, et les propriétés de la matrice chapeau ?
4. Sais-tu résoudre un problème ridge, dire pourquoi il a toujours une solution, et le relier à un prior gaussien ?
5. Sais-tu relier les moindres carrés au bruit gaussien, et décomposer une erreur en biais², variance et bruit ?

**La suite** : le chapitre 9 de workbookIA (overfitting, underfitting et régularisation, avec ses propres implémentations), puis M10, la séparabilité linéaire et la convexité.